In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping_update_calo.df"
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_muon_update_calo.df"
mc_bnb_df = load_df(bnb_path, keys2load, 20)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
mc_bnb_hit0_df.columns

In [ ]:
mc_bnb_hit0_df.index

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

print("data_tot_pot: %.3e" %(data_tot_pot))
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_pfp_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_pfp_df))

In [ ]:
import pandas as pd

# Define the target 4 index levels identifying unique tracks/slices
target_levels = ['__ntuple', 'entry', 'rec.slc..index', 'rec.slc.reco.pfp..index']

# --- 1. Combined Unique Combinations Across hit0 + hit1 + hit2 ---
hit_dfs = {
    '0': mc_bnb_hit0_df,
    '1': mc_bnb_hit1_df,
    '2': mc_bnb_hit2_df,
}

hit_tuple_sets = []
total_hit_rows = 0

for name, df in hit_dfs.items():
    if not df.empty:
        total_hit_rows += len(df)
        # Extract target 4-tuples as a set
        tuples_set = set(
            df.index.to_frame()[target_levels].itertuples(index=False, name=None)
        )
        hit_tuple_sets.append(tuples_set)

# Union of unique 4-tuples across hit0, hit1, and hit2
if hit_tuple_sets:
    combined_hit_unique_tuples = set.union(*hit_tuple_sets)
    n_unique_combined_hits = len(combined_hit_unique_tuples)
else:
    n_unique_combined_hits = 0

print(f"Combined (hit0+hit1+hit2) total hit rows: {total_hit_rows}")
print(f"Combined (hit0+hit1+hit2) unique 4-tuple combinations: {n_unique_combined_hits}")

print("\n" + "=" * 50 + "\n")

# --- 2. Unique Combinations for mc_bnb_pfp_df ---
mc_bnb_pfp_df = mc_bnb_pfp_df.sort_index(level='__ntuple', ascending=True)

pfp_tuples_set = set(
    mc_bnb_pfp_df.index.to_frame()[target_levels].itertuples(index=False, name=None)
)
n_unique_pfp = len(pfp_tuples_set)

print(f"mc_bnb_pfp_df total rows: {len(mc_bnb_pfp_df)}")
print(f"mc_bnb_pfp_df unique 4-tuple combinations: {n_unique_pfp}")

# --- Optional Check: Overlap between Hits and PFP ---
if n_unique_combined_hits > 0 and n_unique_pfp > 0:
    overlap = len(combined_hit_unique_tuples.intersection(pfp_tuples_set))
    print("\n" + "=" * 50 + "\n")
    print(f"Unique tracks present in BOTH PFP and Hits: {overlap}")

In [ ]:
import pandas as pd

# Define your columns
p_type_col = ('pfp', 'trk', 'truth', 'p', 'p_type', '')
weight_col = ('slc', 'wgt', '', '', '', '')  # Optional: set to None if unweighted

# --- 1. Extract and Clean Data ---
valid_mask = mc_bnb_pfp_df[p_type_col].notna()
df_clean = mc_bnb_pfp_df[valid_mask]

if weight_col and weight_col in df_clean.columns:
    weights = df_clean[weight_col].fillna(1.0)
else:
    weights = pd.Series(1.0, index=df_clean.index)

# --- 2. Calculate Weighted & Unweighted Statistics ---
stats_df = pd.DataFrame({
    'p_type': df_clean[p_type_col],
    'weight': weights
})

summary = stats_df.groupby('p_type').agg(
    Counts=('weight', 'count'),
    Weighted_Yield=('weight', 'sum')
).reset_index()

total_counts = summary['Counts'].sum()
total_weighted = summary['Weighted_Yield'].sum()

summary['Raw_%'] = (summary['Counts'] / total_counts) * 100
summary['Weighted_%'] = (summary['Weighted_Yield'] / total_weighted) * 100

# Sort by weighted yield (descending)
summary = summary.sort_values(by='Weighted_Yield', ascending=False)

# --- 3. Pretty Print Output ---
print("\n" + "="*60)
print(f"{'p_type':<15} | {'Counts':<8} | {'Raw %':<8} | {'Weighted':<10} | {'Weighted %':<10}")
print("-" * 60)

for _, row in summary.iterrows():
    print(f"{str(row['p_type']):<15} | {int(row['Counts']):<8d} | {row['Raw_%']:<7.2f}% | {row['Weighted_Yield']:<10.1f} | {row['Weighted_%']:<9.2f}%")

print("-" * 60)
print(f"{'Total':<15} | {total_counts:<8d} | {100.0:<7.2f}% | {total_weighted:<10.1f} | {100.0:<9.2f}%")
print("="*60 + "\n")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Default bin definitions
DEFAULT_BINS_Y = np.linspace(0, 10, 51)  # dE/dx range [MeV/cm]
DEFAULT_BINS_Z = np.linspace(0, 200, 51) # Residual range [cm]


def plot_split_tpc_2d(
    df: pd.DataFrame,
    x_col: str = "rr",
    y_col: str = "dedx",
    x_split_col: str = "x",
    weight_col: str = None,
    bins_x: np.ndarray = DEFAULT_BINS_Z,
    bins_y: np.ndarray = DEFAULT_BINS_Y,
    xlabel: str = "Residual Range [cm]",
    ylabel: str = "dE/dx [MeV/cm]",
    title_prefix: str = "Hit Distribution",
    cmap_name: str = "viridis",
    figsize: tuple = (15, 6),
):
    """Generates a 1x2 2D histogram multiplot split by X < 0 (left) and X >= 0 (right)."""

    # Clean data & extract arrays safely
    mask = df[x_col].notna() & df[y_col].notna() & df[x_split_col].notna()
    plot_df = df[mask]

    x_vals = plot_df[x_col].values
    y_vals = plot_df[y_col].values
    split_vals = plot_df[x_split_col].values

    if weight_col and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0).values
    else:
        weights = np.ones_like(x_vals)

    finite_mask = (
        np.isfinite(x_vals)
        & np.isfinite(y_vals)
        & np.isfinite(split_vals)
        & np.isfinite(weights)
    )
    x_vals, y_vals, split_vals, weights = (
        x_vals[finite_mask],
        y_vals[finite_mask],
        split_vals[finite_mask],
        weights[finite_mask],
    )

    # Subdivide by TPC side using x_split_col
    mask_neg_x = split_vals < 0
    mask_pos_x = split_vals >= 0

    # Setup 1x2 Subplots with shared Y-axis
    fig, (ax_left, ax_right) = plt.subplots(
        1, 2, figsize=figsize, sharey=True, gridspec_kw={"wspace": 0.08}
    )

    cmap = globals().get("sunset_cmap", cmap_name)

    # Calculate global max for uniform colorbar scaling
    h_left, _, _ = np.histogram2d(
        x_vals[mask_neg_x], y_vals[mask_neg_x], bins=[bins_x, bins_y], weights=weights[mask_neg_x]
    )
    h_right, _, _ = np.histogram2d(
        x_vals[mask_pos_x], y_vals[mask_pos_x], bins=[bins_x, bins_y], weights=weights[mask_pos_x]
    )
    vmax = max(h_left.max(), h_right.max())
    vmax = vmax if vmax > 0 else None

    # --- Left Plot: Split Var < 0 ---
    im0 = ax_left.hist2d(
        x_vals[mask_neg_x],
        y_vals[mask_neg_x],
        bins=[bins_x, bins_y],
        weights=weights[mask_neg_x],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    ax_left.set_title(f"{title_prefix}: $X < 0$ cm", fontsize=14, pad=10)
    ax_left.set_xlabel(xlabel, fontsize=14)
    ax_left.set_ylabel(ylabel, fontsize=14)
    ax_left.set_xlim(bins_x[0], bins_x[-1])
    ax_left.set_ylim(bins_y[0], bins_y[-1])
    ax_left.grid(alpha=0.3, linestyle="--")

    # --- Right Plot: Split Var >= 0 ---
    im1 = ax_right.hist2d(
        x_vals[mask_pos_x],
        y_vals[mask_pos_x],
        bins=[bins_x, bins_y],
        weights=weights[mask_pos_x],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    ax_right.set_title(f"{title_prefix}: $X \\geq 0$ cm", fontsize=14, pad=10)
    ax_right.set_xlabel(xlabel, fontsize=14)
    ax_right.set_xlim(bins_x[0], bins_x[-1])
    ax_right.grid(alpha=0.3, linestyle="--")

    # Common Colorbar
    cbar = fig.colorbar(im1, ax=[ax_left, ax_right], pad=0.02)
    cbar.set_label("Weighted Entries", fontsize=12)

    return fig, (ax_left, ax_right)

In [ ]:
# Pass plain string column names
for name, hitdf in hit_dfs.items():
    fig, axes = plot_split_tpc_2d(
        df=hitdf,
        x_col="rr",
        y_col="dedx",
        x_split_col="x",
        weight_col=None,
        bins_x=np.linspace(0, 80, 41),   # Residual Range [cm]
        bins_y=np.linspace(0, 10, 41),    # dE/dx [MeV/cm]
        xlabel="Residual Range [cm]",
        ylabel="Hit dE/dx [MeV/cm]",
        title_prefix=f"Plane {name} dE/dx vs RR",
    )

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_dedx_by_rr_and_tpc(
    df: pd.DataFrame,
    rr_range: tuple = (0.0, 1.0),
    dedx_col: str = "dedx",
    rr_col: str = "rr",
    x_col: str = "x",
    weight_col: str = None,
    bins: np.ndarray = np.linspace(0.0, 10.0, 51),
    x_ranges: list = [(0, 50), (50, 100), (100, 150), (150, 200)],
    stacked: bool = False,
    density: bool = False,
    alpha: float = 0.3,
    linewidth: float = 1.8,
    figsize: tuple = (8, 6),
    title: str = None,
    ax: plt.Axes = None,
):
    # --- 1. Filter RR Range & Clean Data ---
    mask = (
        df[dedx_col].notna()
        & df[rr_col].notna()
        & df[x_col].notna()
        & (df[rr_col] >= rr_range[0])
        & (df[rr_col] < rr_range[1])
    )
    plot_df = df[mask].copy()
    if plot_df.empty:
        raise ValueError(
            f"No valid entries found for {rr_col} in range {rr_range}."
        )

    # Resolve Weights
    if weight_col is not None and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0)
    else:
        weights = pd.Series(1.0, index=plot_df.index)

    # --- 2. Categorization by |x| distance range (combines both TPCs) ---
    abs_x = plot_df[x_col].abs()
    group_masks = [
        (abs_x >= lo) & (abs_x < hi) for lo, hi in x_ranges
    ]

    grouped_data = [plot_df.loc[m, dedx_col] for m in group_masks]
    grouped_weights = [weights[m] for m in group_masks]

    # --- Handle Normalization (density=True) ---
    bin_width = bins[1] - bins[0]
    if density:
        if stacked:
            # Normalize so the combined stacked area sums to 1.0
            total_weight = sum(w.sum() for w in grouped_weights)
            if total_weight > 0:
                scale = 1.0 / (total_weight * bin_width)
                grouped_weights = [w * scale for w in grouped_weights]
        else:
            # Normalize each subgroup independently so each group's area sums to 1.0
            normed_weights = []
            for w in grouped_weights:
                w_sum = w.sum()
                if w_sum > 0:
                    normed_weights.append(w / (w_sum * bin_width))
                else:
                    normed_weights.append(w)
            grouped_weights = normed_weights

    # Colors/labels auto-generated for however many x_ranges are given
    default_colors = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]
    colors = [default_colors[i % len(default_colors)] for i in range(len(x_ranges))]
    labels = [rf"${lo} \leq |x| < {hi}$ cm" for lo, hi in x_ranges]

    # --- 3. Plotting ---
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()

    # Layer 1: Filled steps
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="stepfilled",
        color=colors,
        alpha=alpha,
        label=labels,
    )
    # Layer 2: Outlines
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="step",
        color=colors,
        linewidth=linewidth,
    )

    # --- 4. Styling & Formatting ---
    ax.set_xlabel(r"Hit $dE/dx$ [MeV/cm]", fontsize=14)
    if density:
        ax.set_ylabel("A.U.", fontsize=14)
    elif weight_col:
        ax.set_ylabel("Weighted Hits", fontsize=14)
    else:
        ax.set_ylabel("Hits", fontsize=14)

    if title is None:
        title = rf"Hit $dE/dx$ Distribution ({rr_range[0]} $\leq$ RR < {rr_range[1]} cm)"
    ax.set_title(title, fontsize=15, pad=12)

    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
    ax.tick_params(axis="both", which="both", labelsize=12, direction="in")
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)
    ax.legend(
        fontsize=12,
        frameon=True,
        framealpha=1.0,
        edgecolor="black",
        fancybox=False,
    )
    return fig, ax

In [ ]:
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(2, 3), (6, 7), (15, 16)]

for rr_min, rr_max in rr_ranges:
    fig, axes = plt.subplots(
        1, len(hit_dfs), figsize=(18, 5), sharey=True, gridspec_kw={"wspace": 0.08}
    )

    max_y_value = 0  # Track global maximum density across planes

    for plane_idx, (df_plane, ax) in enumerate(zip(hit_dfs, axes)):
        try:
            plot_dedx_by_rr_and_tpc(
                df=df_plane,
                rr_range=(rr_min, rr_max),
                dedx_col="dedx",
                rr_col="rr",
                x_col="x",
                weight_col=None,
                bins=np.linspace(0.0, 10.0, 51),
                stacked=False,
                density=True,  # 👈 Now supported!
                title=f"Plane {plane_idx}",
                ax=ax,
            )

            # Record maximum bin height in this subplot
            current_max = ax.get_ylim()[1] / 1.15
            if current_max > max_y_value:
                max_y_value = current_max

        except ValueError:
            ax.set_title(f"Plane {plane_idx}: No Data", fontsize=15, pad=12)
            continue

        # Clean up side subplots
        if plane_idx > 0:
            ax.set_ylabel("")
            legend = ax.get_legend()
            if legend:
                legend.remove()

    # Apply the global max limit + 15% headroom to all subplots
    if max_y_value > 0:
        axes[0].set_ylim(0, max_y_value * 1.15)

    fig.suptitle(
        rf"Normalized Hit $dE/dx$ Comparison ({rr_min} $\leq$ RR < {rr_max} cm)",
        fontsize=16,
        y=1.03,
    )

    plt.show()

In [ ]:
hfit = load_physics_classes()

In [ ]:
pdg = 13
particle="muon"
if "pion" in bnb_path:
    pdg = 211
    particle = "pion"
    

all_results, fit_params = analyze_theoretical(hit_dfs,hfit, pdg,  particle=particle)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d


def find_subbin_peak(counts, edges, smooth_sigma=2.0):
    """Finds sub-bin peak coordinates and propagates Poisson uncertainty to calculate

    the uncertainty in peak position (x_err).
    """
    if len(counts) == 0 or counts.max() == 0:
        return 0.0, 0.0, 0.0

    bin_width = edges[1] - edges[0]
    bin_centers = 0.5 * (edges[:-1] + edges[1:])

    # 1. Smooth out statistical noise spikes
    smoothed = gaussian_filter1d(counts.astype(float), sigma=smooth_sigma)
    idx = np.argmax(smoothed)

    # 2. Sub-bin parabolic interpolation with error propagation
    if 0 < idx < len(counts) - 1:
        y1, y2, y3 = smoothed[idx - 1], smoothed[idx], smoothed[idx + 1]
        denom = y1 - 2 * y2 + y3

        if denom != 0:
            delta = 0.5 * (y1 - y3) / denom
            x_peak = bin_centers[idx] + delta * bin_width
            y_peak = y2 - 0.25 * (y1 - y3) * delta

            # Variance propagation assuming Poisson errors on original bin counts
            var1, var2, var3 = (
                max(counts[idx - 1], 1.0),
                max(counts[idx], 1.0),
                max(counts[idx + 1], 1.0),
            )
            d_delta_dy1 = (y3 - y2) / (denom**2)
            d_delta_dy2 = (y1 - y3) / (denom**2)
            d_delta_dy3 = (y2 - y1) / (denom**2)

            var_delta = (
                (d_delta_dy1**2) * var1
                + (d_delta_dy2**2) * var2
                + (d_delta_dy3**2) * var3
            )
            x_err = bin_width * np.sqrt(var_delta)

            return x_peak, y_peak, x_err

    # Fallback to single bin width error
    return bin_centers[idx], counts[idx], bin_width / np.sqrt(12)



def plot_slice_diagnostic_max(
    df,
    plane,
    tpc,
    rr_min,
    rr_max,
    dedx_col="dedx",
    nbins=100,
    hist_xmin=0.0,
    hist_xmax=10.0,
    smooth_sigma=2.0,
):
    """Sanity-check plot for a single (plane, tpc, rr-range) slice: plots the raw

    dE/dx histogram and overlays the continuous envelope peak found beyond bin
    resolution.
    """
    sl = df if tpc == -1 else df[df["tpc"] == tpc]
    sl = sl[(sl["rr"] >= rr_min) & (sl["rr"] < rr_max) & (sl["pitch"] <= 2)]

    fig, ax = plt.subplots(figsize=(8, 5.5))

    # Plot raw data histogram
    counts, edges, _ = ax.hist(
        sl[dedx_col].dropna(),
        bins=nbins,
        range=(hist_xmin, hist_xmax),
        histtype="stepfilled",
        alpha=0.3,
        color="steelblue",
        edgecolor="navy",
        label=f"data (N={len(sl)})",
    )

    bin_width = (hist_xmax - hist_xmin) / nbins
    max_x, max_y = 0.0, 0.0

    # Calculate and overlay continuous envelope peak
    if len(counts) > 0 and counts.max() > 0:
        max_x, max_y, x_err = find_subbin_peak(
            counts, edges, smooth_sigma=smooth_sigma
        )

        # Vertical reference line at the continuous peak X value
        ax.axvline(
            x=max_x,
            color="crimson",
            linestyle="--",
            linewidth=2,
            label=f"Peak X = {max_x:.2f} MeV/cm",
        )

        # Point marker at (sub-bin X, continuous peak Y)
        ax.plot(
            max_x,
            max_y,
            "ro",
            markersize=7,
            label=f"Peak Count = {max_y:.0f}",
        )

        # Direct annotation pointing to the continuous peak
        ax.annotate(
            f"Max: {max_y:.0f}\n@ {max_x:.2f} MeV/cm",
            xy=(max_x, max_y),
            xytext=(
                max_x + 0.08 * (hist_xmax - hist_xmin),
                max_y * 0.85,
            ),
            arrowprops=dict(
                facecolor="crimson", shrink=0.05, width=1.5, headwidth=6
            ),
            fontsize=9,
            fontweight="bold",
        )

    # Plot formatting
    ax.set_ylim(0, max_y * 1.20 if max_y > 0 else 1.0)
    ax.set_xlabel(r"$dE/dx$ [MeV/cm]")
    ax.set_ylabel(f"hits / {bin_width:.2f} MeV/cm")

    if tpc == -1:
        ax.set_title(
            f"plane {plane}, TPCs Combined, {rr_min:g} <= rr < {rr_max:g} cm"
        )
    else:
        ax.set_title(
            f"plane {plane}, tpc {tpc}, {rr_min:g} <= rr < {rr_max:g} cm"
        )

    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()

    return fig, max_x, max_y

In [ ]:
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(1, 2), (2, 3), (3,4), (4, 5), (6, 7), (15, 16), (30, 31)]

import os
# Define output path
output_dir = "/exp/sbnd/data/users/lpelegri/TLEGraphs/dEdxPDF"
os.makedirs(output_dir, exist_ok=True)

plane = 2
df =  hit_dfs[plane]
tpc = -1
for rr_min, rr_max in rr_ranges:

    fig_1 = plot_slice_diagnostic_max(
        df,
        plane,
        tpc,
        rr_min,
        rr_max,
        dedx_col="dedx",
        nbins=100,
        hist_xmin=0.0,
        hist_xmax=10.0,
    )

In [ ]:


def compute_shift_diagnostics(
    df,
    plane,
    tpc,
    hfit,
    pdg,
    dedx_col="dedx",
    rr_min=2.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    pitch=0.55,
    mass=None,
    min_entries=200,
    hist_nbins=100,
    hist_xmin=0.0,
    hist_xmax=10.0,
    smooth_sigma=2.0,
    verbose=False,
):
    """Computes difference between data histogram max and theoretical MPV across

    rr slices with error estimation.
    """
    sl_all = df if tpc == -1 else df[df["tpc"] == tpc]
    edges = make_rr_edges(rr_min, rr_max, rr_bin_width)
    rows = []

    for lo, hi in zip(edges[:-1], edges[1:]):
        sl = sl_all[(sl_all["rr"] >= lo) & (sl_all["rr"] < hi)]
        if len(sl) < min_entries:
            continue
        rr_center = 0.5 * (lo + hi)

        pdf = build_theoretical_pdf(hfit, pdg, rr_center, pitch, mass=mass)
        theory_mpv = robust_max_x_py(pdf, 0.0, 10.0, 2000)

        # Histogram peak extraction with estimated uncertainty
        data = sl[dedx_col].dropna().to_numpy()
        counts, h_edges = np.histogram(
            data, bins=hist_nbins, range=(hist_xmin, hist_xmax)
        )
        hist_max, _, hist_max_err = find_subbin_peak(
            counts, h_edges, smooth_sigma=smooth_sigma
        )

        diff_hist_theory = hist_max - theory_mpv

        if verbose:
            print(
                f"  rr={rr_center:.2f}: theory_mpv={theory_mpv:.3f}, "
                f"hist_max={hist_max:.3f} +/- {hist_max_err:.3f}, "
                f"diff_hist_theory={diff_hist_theory:.3f}"
            )

        rows.append(
            dict(
                plane=plane,
                tpc=tpc,
                rr_center=rr_center,
                n_hits=len(sl),
                theory_mpv=theory_mpv,
                hist_max=hist_max,
                hist_max_err=hist_max_err,
                diff_hist_theory=diff_hist_theory,
                diff_err=hist_max_err,  # Uncertainty carries directly into difference
            )
        )

    return pd.DataFrame(rows)


def plot_shift_diagnostics(
    diag_df,
    plane,
    tpc,
    pitch=None,
    out_prefix="shift_diag",
    fit_diff_hist_theory_curve=True,
):
    """Plots diff_hist_theory with error bars vs rr_center and overlays optional fit curve."""
    if pitch is not None:
        print(f"theoretical MPV computed at pitch = {pitch:.2f} cm")

    fig, ax = plt.subplots(figsize=(8, 5.5))

    ax.axhline(0, color="gray", lw=1, linestyle=":")

    # Plot histogram max - theory with error bars
    ax.errorbar(
        diag_df["rr_center"],
        diag_df["diff_hist_theory"],
        yerr=diag_df["diff_err"],
        fmt="s-",
        color="mediumpurple",
        ms=4,
        capsize=3,
        linewidth=1.2,
        label=r"hist. max $-$ theory  (data peak vs. unsmeared theory)",
    )

    if fit_diff_hist_theory_curve:
        popt, perr = fit_diff_hist_theory(diag_df)
        if popt is not None:
            a, b, c = popt
            a_e, b_e, c_e = perr
            print(
                f"diff_hist_theory fit: a={a:.4f}+/-{a_e:.4f}, "
                f"b={b:.4f}+/-{b_e:.4f}, c={c:.4f}+/-{c_e:.4f}"
            )
            xs_fit = np.linspace(
                diag_df["rr_center"].min(), diag_df["rr_center"].max(), 200
            )
            ys_fit = exp_decay_plateau(xs_fit, *popt)
            ax.plot(
                xs_fit,
                ys_fit,
                "--",
                color="darkindigo" if "darkindigo" in plt.cm.datad else "purple",
                lw=1.8,
                alpha=0.8,
                label=rf"fit: ${a:.3f} + {b:.3f}\,e^{{-rr/{c:.3f}}}$",
            )
        else:
            print(
                "diff_hist_theory fit failed or too few points -- skipping overlay"
            )

    ax.set_xlabel("rr [cm]")
    ax.set_ylabel(r"$\Delta$ MPV [MeV/cm]")
    tpc_label = "combined" if tpc == -1 else tpc
    title = f"plane {plane}, tpc {tpc_label}"
    if pitch is not None:
        title += f", pitch={pitch:.2f} cm"
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle=":", alpha=0.5)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_p{plane}_t{tpc}.pdf")
    plt.show()
    return fig

def fit_diff_hist_theory(diag_df, x_col="rr_center", y_col="diff_hist_theory", p0=None):
    """Fits exp_decay_plateau to diff_hist_theory vs rr_center.
    Returns (popt, perr) or (None, None) if the fit fails or there
    aren't enough points."""
    xs = diag_df[x_col].to_numpy()
    ys = diag_df[y_col].to_numpy()
    mask = np.isfinite(xs) & np.isfinite(ys)
    xs, ys = xs[mask], ys[mask]

    if len(xs) < 4:
        return None, None

    if p0 is None:
        # == Rough starting guess: plateau ~ value at largest rr,
        # == amplitude ~ (value at smallest rr) - plateau, decay length
        # == ~ a third of the rr range
        order = np.argsort(xs)
        a0 = ys[order][-1]
        b0 = ys[order][0] - a0
        c0 = max((xs.max() - xs.min()) / 3.0, 1e-3)
        p0 = [a0, b0, c0]

    try:
        popt, pcov = curve_fit(exp_decay_plateau, xs, ys, p0=p0, maxfev=20000)
        perr = np.sqrt(np.diag(pcov))
        return popt, perr
    except Exception:
        return None, None

In [ ]:

diag_df_032 = compute_shift_diagnostics(
    hit_dfs[plane], plane, tpc, hfit, pdg=pdg,
    pitch=0.32, verbose=False,
)
plot_shift_diagnostics(diag_df_032, plane, tpc, pitch=0.32, out_prefix="shift_diag_pitch032")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

PLANE_COLORS = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}

TPC_STYLES = {
    0: {"linestyle": "--", "marker": "o", "label_prefix": "x < 0 (TPC 0)"},
    1: {"linestyle": ":", "marker": "s", "label_prefix": "x > 0 (TPC 1)"},
    -1: {"linestyle": "-", "marker": "^", "label_prefix": "Combined"},
}


def exp_decay_plateau(x, a, b, c):
    """Exponential decay model with offset plateau: a + b * exp(-x / c)."""
    return a + b * np.exp(-x / c)


def fit_diff_hist_theory(diag_df):
    """Fits an exponential decay with plateau to diff_hist_theory vs rr_center,

    using bounded optimization to prevent parameter collapse.
    """
    if len(diag_df) < 3:
        return None, None
    try:
        x = diag_df["rr_center"].values
        y = diag_df["diff_hist_theory"].values

        # Dynamic initial guesses based on data range
        a_0 = np.median(y[-5:])  # Plateau level at high rr
        b_0 = y[0] - a_0  # Amplitude drop at low rr (typically negative)
        c_0 = 3.0  # Typical decay length scale (~3 cm)

        p0 = [a_0, b_0, c_0]

        # Bounds: [a_min, b_min, c_min], [a_max, b_max, c_max]
        # Restricts c to a physical decay scale between 0.1 cm and 50 cm
        bounds = ([-1.0, -10.0, 0.1], [1.0, 10.0, 50.0])

        sigma = (
            diag_df["diff_err"].values
            if "diff_err" in diag_df
            else diag_df.get("hist_max_err", None)
        )

        popt, pcov = curve_fit(
            exp_decay_plateau,
            x,
            y,
            p0=p0,
            bounds=bounds,
            sigma=sigma,
            absolute_sigma=True if sigma is not None else False,
            maxfev=10000,
        )

        # Check if scipy failed to estimate parameter covariance matrix
        if np.any(np.isinf(pcov)) or np.any(np.isnan(pcov)):
            return None, None

        perr = np.sqrt(np.diag(pcov))
        return popt, perr
    except Exception:
        return None, None


def analyze_shift_diagnostics(
    hit_dfs,
    hfit,
    pdg,
    particle="muon",
    dedx_col="dedx",
    rr_max_by_particle=None,
    pitch=0.32,
    mass=None,
    rr_min=3.0,
    rr_bin_width=1.0,
    min_entries=200,
    hist_nbins=100,
    hist_xmin=0.0,
    hist_xmax=10.0,
    smooth_sigma=2.0,
    out_prefix="shift_diag",
    verbose=True,
):
    """Runs compute_shift_diagnostics across planes and TPCs (0, 1, -1) and fits

    exponential decay plateau curves to the resulting shift trends vs. rr_center.
    """
    if rr_max_by_particle is None:
        rr_max_by_particle = {"muon": 80.0, "pion": 40.0, "proton": 60.0}
    rr_max = rr_max_by_particle.get(particle, 80.0)

    all_results = {}
    fit_params = {}

    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1, -1):
            if verbose:
                tpc_label = "combined" if tpc == -1 else tpc
                print(f"[shift_diag] Plane {plane}, TPC {tpc_label}")

            res = compute_shift_diagnostics(
                df,
                plane=plane,
                tpc=tpc,
                hfit=hfit,
                pdg=pdg,
                dedx_col=dedx_col,
                rr_min=rr_min,
                rr_max=rr_max,
                rr_bin_width=rr_bin_width,
                pitch=pitch,
                mass=mass,
                min_entries=min_entries,
                hist_nbins=hist_nbins,
                hist_xmin=hist_xmin,
                hist_xmax=hist_xmax,
                smooth_sigma=smooth_sigma,
                verbose=verbose,
            )
            all_results[(plane, tpc)] = res

            popt = perr = None
            if len(res) >= 3:
                popt, perr = fit_diff_hist_theory(res)
                if popt is None and verbose:
                    print("  exp decay fit failed")
            fit_params[(plane, tpc)] = (popt, perr)

    non_empty = [
        r.assign(plane=p, tpc=t)
        for (p, t), r in all_results.items()
        if len(r)
    ]
    combined = (
        pd.concat(non_empty, ignore_index=True)
        if non_empty
        else pd.DataFrame()
    )

    if len(combined):
        combined.to_hdf(
            f"{out_prefix}_slices.h5",
            key="fits",
            mode="w",
            format="table",
            complib="blosc",
            complevel=9,
        )

    return all_results, fit_params


def plot_all_planes(
    all_results, fit_params, particle="muon", out_prefix="shift_diag"
):
    """Plots all (plane, tpc) shift fit curves on a single square plot."""
    fig, ax = plt.subplots(figsize=(8, 8))

    for (plane, tpc), res in all_results.items():
        popt, _ = fit_params.get((plane, tpc), (None, None))
        if popt is None or res is None or not len(res):
            continue

        color = PLANE_COLORS.get(plane, "black")
        style_info = TPC_STYLES.get(
            tpc,
            {"linestyle": "-", "label_prefix": f"tpc={tpc}"},
        )
        style = style_info["linestyle"]
        prefix = style_info["label_prefix"]

        xs = np.linspace(res["rr_center"].min(), res["rr_center"].max(), 200)
        a, b, c = popt
        label = f"{prefix}, Plane {plane}: ${a:.3f} + {b:.3f}e^{{-rr/{c:.2f}}}$"
        ax.plot(
            xs,
            exp_decay_plateau(xs, *popt),
            linestyle=style,
            color=color,
            linewidth=2.0,
            label=label,
        )

    ax.set_xlabel("rr [cm]", fontsize=12)
    ax.set_ylabel(r"$\Delta$ MPV (hist. max $-$ theory) [MeV/cm]", fontsize=12)

    ax.legend(fontsize=9, loc="lower right", framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes.pdf")
    plt.show()


def plot_by_plane(
    all_results,
    fit_params,
    particle="muon",
    out_prefix="shift_diag",
    x_limits=None,
):
    """One square subplot per plane with TPC 0, 1, and Combined overlaid side by side."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    tpc_colors = {0: "tab:red", 1: "tab:blue", -1: "tab:green"}

    for plane in range(3):
        ax = axes[plane]
        y_visible_min, y_visible_max = np.inf, -np.inf

        for tpc in [0, 1, -1]:
            res = all_results.get((plane, tpc))
            popt, _ = fit_params.get((plane, tpc), (None, None))

            if res is None or not len(res):
                continue

            color = tpc_colors[tpc]
            style_info = TPC_STYLES[tpc]
            style = style_info["linestyle"]
            marker = style_info["marker"]
            prefix = style_info["label_prefix"]

            if x_limits is not None:
                mask = (res["rr_center"] >= x_limits[0]) & (
                    res["rr_center"] <= x_limits[1]
                )
                res_filtered = res[mask]
            else:
                res_filtered = res

            # Plot data points with error bars
            if len(res_filtered):
                yerr = (
                    res_filtered["diff_err"]
                    if "diff_err" in res_filtered
                    else res_filtered["hist_max_err"]
                )
                ax.errorbar(
                    res_filtered["rr_center"],
                    res_filtered["diff_hist_theory"],
                    yerr=yerr,
                    fmt=marker,
                    color=color,
                    ms=4,
                    capsize=2,
                    label=f"{prefix} Data",
                )
                if x_limits is not None:
                    y_visible_min = min(
                        y_visible_min,
                        (res_filtered["diff_hist_theory"] - yerr).min(),
                    )
                    y_visible_max = max(
                        y_visible_max,
                        (res_filtered["diff_hist_theory"] + yerr).max(),
                    )

            # Plot fit curve
            if popt is not None:
                x_min = x_limits[0] if x_limits else res["rr_center"].min()
                x_max = x_limits[1] if x_limits else res["rr_center"].max()
                xs = np.linspace(x_min, x_max, 200)
                ys = exp_decay_plateau(xs, *popt)

                a, b, c = popt
                label_fit = f"{prefix}: ${a:.3f} + {b:.3f}e^{{-rr/{c:.2f}}}$"
                ax.plot(
                    xs,
                    ys,
                    linestyle=style,
                    color=color,
                    linewidth=1.8,
                    label=label_fit,
                )

                if x_limits is not None:
                    y_visible_min = min(y_visible_min, ys.min())
                    y_visible_max = max(y_visible_max, ys.max())

        if x_limits is not None:
            ax.set_xlim(x_limits)
            if not np.isinf(y_visible_min) and not np.isinf(y_visible_max):
                y_pad = (y_visible_max - y_visible_min) * 0.1
                ax.set_ylim(y_visible_min - y_pad, y_visible_max + y_pad)

        ax.set_title(f"Plane {plane}", fontsize=14, pad=10)
        ax.set_xlabel("rr [cm]", fontsize=12)
        ax.set_ylabel(r"$\Delta$ MPV [MeV/cm]", fontsize=12)

        ax.legend(fontsize=9, loc="lower right", framealpha=0.9)
        ax.grid(True, linestyle=":", alpha=0.5)
        ax.set_box_aspect(1)

    fig.tight_layout()
    zoom_suffix = (
        f"_zoom_{x_limits[0]}_{x_limits[1]}".replace(".", "p")
        if x_limits
        else ""
    )
    fig.savefig(f"{out_prefix}_by_plane{zoom_suffix}.pdf")
    plt.show()

In [ ]:
# 1. Organize your hit DataFrames by plane index
# Each DataFrame must contain columns: 'dedx', 'rr', and 'tpc' (0 or 1)
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]

# 2. Run the main analysis wrapper across all planes and TPC combinations (0, 1, -1)
all_results, fit_params = analyze_shift_diagnostics(
    hit_dfs=hit_dfs,
    hfit=hfit,  # Theoretical Bethe-Bloch / Landau-Vavilov model fit
    pdg=pdg,  # Particle PDG (e.g., 13 for muon)
    particle=particle,  # Sets default rr_max threshold (80 cm for muons)
    dedx_col="dedx",
    pitch=0.32,  # Track pitch in cm
    rr_min=3.0,  # Minimum residual range cut (cm)
    rr_bin_width=1.0,  # Bin width for rr slices (cm)
    hist_nbins=100,  # Bin count for dE/dx peak extraction
    out_prefix="",  # Prefix for HDF5 output and saved PDFs
    verbose=False,
)

# 3. Plot all 9 fit curves (3 planes x 3 TPC configs) on a single square canvas
plot_all_planes(
    all_results=all_results,
    fit_params=fit_params,
    particle="muon",
    out_prefix="muon_shift_diag",
)

# 4. Generate side-by-side subplots (one per plane) showing data error bars and fits
plot_by_plane(
    all_results=all_results,
    fit_params=fit_params,
    particle="muon",
    out_prefix="muon_shift_diag",
    x_limits=(3.0, 40.0),  # Optional: set custom range for x-axis zoom
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import matplotlib.pyplot as plt
import numpy as np


def exp_decay_plateau(x, a, b, c):
    """Exponential decay model with offset plateau: a + b * exp(-x / c)."""
    return a + b * np.exp(-x / c)


def plot_slice_diagnostic_theory_only(
    df,
    plane,
    tpc,
    rr_min,
    rr_max,
    hfit,
    pdg=13,
    mass=None,
    fit_params=None,
    dedx_col="dedx",
    nbins=100,
    hist_xmin=0.0,
    hist_xmax=10.0,
    smooth_sigma=2.0,
    xlim=None,  # e.g., xlim=(1.5, 3.5)
):
    """Plots the raw dE/dx histogram with sub-bin data peak extraction, overlaid

    with unsmeared and shift-corrected theoretical curves.
    """
    # 1. Filter dataset by residual range, pitch, and TPC configuration
    mask = (df["rr"] >= rr_min) & (df["rr"] < rr_max) & (df["pitch"] <= 2)
    if tpc != -1:
        mask &= df["tpc"] == tpc

    sl = df[mask]
    rr_center = 0.5 * (rr_min + rr_max)

    fig, ax = plt.subplots(figsize=(8, 5.5))

    # 2. Plot raw data histogram
    counts, edges, _ = ax.hist(
        sl[dedx_col].dropna(),
        bins=nbins,
        range=(hist_xmin, hist_xmax),
        histtype="stepfilled",
        alpha=0.3,
        color="steelblue",
        edgecolor="navy",
        label=f"data (N={len(sl)})",
    )
    bin_width = (hist_xmax - hist_xmin) / nbins
    bin_centers = 0.5 * (edges[:-1] + edges[1:])
    x_eval = np.linspace(hist_xmin, hist_xmax, 400)
    data_max = counts.max() if len(counts) > 0 else 0.0

    # 3. Find histogram peak and add peak reference line
    max_x, max_y, x_err = 0.0, 0.0, 0.0
    if len(counts) > 0 and data_max > 0:
        max_x, max_y, x_err = find_subbin_peak(
            counts, edges, smooth_sigma=smooth_sigma
        )

        ax.axvline(
            x=max_x,
            color="crimson",
            linestyle="--",
            linewidth=1.8,
            label=rf"data max = {max_x:.2f} $\pm$ {x_err:.2f} MeV/cm",
        )
        ax.plot(max_x, max_y, "ro", markersize=6)

    # 4. Build unsmeared theoretical PDF
    mean_pitch =  0.32

    pdf = build_theoretical_pdf(hfit, pdg, rr_center, mean_pitch, mass=mass)
    theo_mpv = pdf.GetMaximumX()

    norm = len(sl) * bin_width
    y_theo = np.array([pdf.Eval(xv) * norm for xv in x_eval])

    if y_theo.max() > 0 and data_max > 0:
        y_theo = y_theo * (data_max / y_theo.max())

    curve_max = max(data_max, y_theo.max()) if len(y_theo) > 0 else 1.0

    ax.plot(
        x_eval,
        y_theo,
        "r:",
        lw=2.5,
        label=f"theory, unsmeared (MPV={theo_mpv:.2f})",
    )

    # 5. Evaluate and plot shifted theoretical PDF
    if fit_params is not None:
        popt, _ = fit_params.get((plane, tpc), (None, None))
        if popt is not None:
            delta_mpv = exp_decay_plateau(rr_center, *popt)
            shifted_mpv = theo_mpv + delta_mpv

            y_shifted = np.array(
                [pdf.Eval(xv - delta_mpv) * norm for xv in x_eval]
            )
            if y_shifted.max() > 0 and data_max > 0:
                y_shifted = y_shifted * (data_max / y_shifted.max())

            ax.plot(
                x_eval,
                y_shifted,
                "-",
                color="purple",
                lw=2.0,
                label=rf"theory shifted ($\Delta$MPV={delta_mpv:+.2f}, MPV={shifted_mpv:.2f})",
            )
            curve_max = max(curve_max, y_shifted.max())

    # 6. Zoom limits and axis scaling
    if xlim is not None:
        ax.set_xlim(xlim)
        mask_zoom = (bin_centers >= xlim[0]) & (bin_centers <= xlim[1])
        if np.any(mask_zoom):
            visible_max = counts[mask_zoom].max()
            ax.set_ylim(0, max(visible_max, curve_max) * 1.25)
    else:
        ax.set_ylim(0, curve_max * 1.15 if curve_max > 0 else 1.0)

    # Plot formatting
    ax.set_xlabel(r"$dE/dx$ [MeV/cm]")
    ax.set_ylabel(f"hits / {bin_width:.2f} MeV/cm")
    tpc_label = "all" if tpc == -1 else str(tpc)
    ax.set_title(
        f"plane {plane}, tpc {tpc_label}, {rr_min:g} $\\leq$ rr < {rr_max:g} cm"
    )
    ax.legend(fontsize=9, loc="upper right")
    ax.grid(True, linestyle=":", alpha=0.5)
    fig.tight_layout()

    return fig

In [ ]:
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(1, 2), (2, 3), (3,4), (4, 5), (6, 7), (15, 16), (30, 31)]

import os
# Define output path
output_dir = "/exp/sbnd/data/users/lpelegri/TLEGraphs/dEdxPDF"
os.makedirs(output_dir, exist_ok=True)

plane = 2
df =  hit_dfs[plane]

for rr_min, rr_max in rr_ranges:

    # 2. Theory-Only Plot
    fig2 = plot_slice_diagnostic_theory_only(
        df,
        plane,
        -1,
        rr_min,
        rr_max,
        hfit=hfit,
        pdg=pdg,
        fit_params=fit_params,
        dedx_col="dedx",
        nbins=100,
        hist_xmin=0.0,
        hist_xmax=10.0
    )

    fname2 = f"diag_theory_p2_tpc0_rr_{rr_min:g}_{rr_max:g}.png"
    fig2.savefig(
        os.path.join(output_dir, fname2), dpi=300, bbox_inches="tight"
    )
    

In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import curve_fit
from scipy.signal import fftconvolve
'''

def convolved_theoretical_pdf(
    x_query, sigma, shift, x_grid, pdf_vals, dx, area=1.0
):
    """2-parameter model: amplitude is fixed to the slice area while fitting sigma and shift."""
    x_centered = x_grid - x_grid[len(x_grid) // 2]
    kernel = gaussian_kernel(x_centered, sigma)
    kernel = kernel / (kernel.sum() * dx)  # Normalize Gaussian kernel
    conv = fftconvolve(pdf_vals, kernel, mode="same") * dx
    return area * np.interp(x_query - shift, x_grid, conv)


def fit_theoretical_conv_slice(
    bin_centers,
    bin_counts,
    bin_errs,
    x_grid,
    pdf_vals,
    dx,
    mpv_theory,
    fit_range=None,
    sigma0=0.2,
    shift0=0.0,
    sigma_bounds=(1e-4, 5.0),
    shift_bounds=(-3.0, 3.0),
):
    """Fits only (sigma, shift) using scipy curve_fit."""
    if fit_range is None:
        fit_range = (0.7 * mpv_theory, 1.5 * mpv_theory)

    mask = (
        (bin_centers >= fit_range[0])
        & (bin_centers <= fit_range[1])
        & (bin_counts > 0)
    )
    if mask.sum() < 5:
        return None

    xs, ys, yerr = bin_centers[mask], bin_counts[mask], bin_errs[mask]
    yerr = np.where(yerr > 0, yerr, 1.0)

    # Pre-compute target area from slice bin spacing to fix total normalization
    bin_w = bin_centers[1] - bin_centers[0]
    slice_area = np.sum(ys) * bin_w

    def model(x, sigma, shift):
        return convolved_theoretical_pdf(
            x, sigma, shift, x_grid, pdf_vals, dx, area=slice_area
        )

    try:
        popt, pcov = curve_fit(
            model,
            xs,
            ys,
            p0=[sigma0, shift0],
            sigma=yerr,
            absolute_sigma=True,
            bounds=(
                [sigma_bounds[0], shift_bounds[0]],
                [sigma_bounds[1], shift_bounds[1]],
            ),
            maxfev=20000,
        )
    except Exception:
        return None

    perr = np.sqrt(np.diag(pcov))
    return dict(
        sigma=popt[0],
        sigma_err=perr[0],
        shift=popt[1],
        shift_err=perr[1],
        fit_range=fit_range,
    )
'''

def convolved_theoretical_pdf(
    x_query, sigma, shift, x_grid, pdf_vals, dx, target_peak=1.0
):
    """2-parameter model: sigma and shift are fit; the curve is rescaled so
    its own peak height matches `target_peak` (the data's peak height),
    rather than matching the data's total area.
    """
    x_centered = x_grid - x_grid[len(x_grid) // 2]
    kernel = gaussian_kernel(x_centered, sigma)
    kernel = kernel / (kernel.sum() * dx)  # Normalize Gaussian kernel
    conv = fftconvolve(pdf_vals, kernel, mode="same") * dx

    conv_peak = conv.max()
    if conv_peak <= 0 or not np.isfinite(conv_peak):
        conv_peak = 1.0  # guard against degenerate curves during fit iterations

    scale = target_peak / conv_peak
    return scale * np.interp(x_query - shift, x_grid, conv)


def fit_theoretical_conv_slice(
    bin_centers,
    bin_counts,
    bin_errs,
    x_grid,
    pdf_vals,
    dx,
    mpv_theory,
    fit_range=None,
    sigma0=0.2,
    shift0=0.0,
    sigma_bounds=(1e-4, 5.0),
    shift_bounds=(-3.0, 3.0),
):
    """Fits only (sigma, shift) using scipy curve_fit.

    The model's amplitude is not a free fit parameter: at every evaluation,
    the convolved curve is rescaled so its peak height matches the data's
    peak height within the fit range (rather than matching total area).
    """
    if fit_range is None:
        fit_range = (0.7 * mpv_theory, 1.4 * mpv_theory)
    mask = (
        (bin_centers >= fit_range[0])
        & (bin_centers <= fit_range[1])
        & (bin_counts > 0)
    )
    if mask.sum() < 5:
        return None
    xs, ys, yerr = bin_centers[mask], bin_counts[mask], bin_errs[mask]
    yerr = np.where(yerr > 0, yerr, 1.0)

    # Target peak height = data's max count within the fit range
    target_peak = ys.max()

    def model(x, sigma, shift):
        return convolved_theoretical_pdf(
            x, sigma, shift, x_grid, pdf_vals, dx, target_peak=target_peak
        )

    try:
        popt, pcov = curve_fit(
            model,
            xs,
            ys,
            p0=[sigma0, shift0],
            sigma=yerr,
            absolute_sigma=True,
            bounds=(
                [sigma_bounds[0], shift_bounds[0]],
                [sigma_bounds[1], shift_bounds[1]],
            ),
            maxfev=20000,
        )
    except Exception:
        return None
    perr = np.sqrt(np.diag(pcov))
    return dict(
        sigma=popt[0],
        sigma_err=perr[0],
        shift=popt[1],
        shift_err=perr[1],
        fit_range=fit_range,
    )


import numpy as np
from scipy.optimize import curve_fit

def fit_sigma_vs_mpv(
    res_df,
    x_col="mpv_x",
    y_col="gsigma",
    yerr_col="gsigma_err",
    max_yerr=1.0,  # Relaxed default threshold
):
    # Filter valid finite points
    mask = (
        (res_df[yerr_col] <= max_yerr) & 
        (res_df[yerr_col] > 0) & 
        np.isfinite(res_df[yerr_col]) &
        np.isfinite(res_df[x_col]) &
        np.isfinite(res_df[y_col])
    )
    filtered_df = res_df[mask].sort_values(x_col)

    if len(filtered_df) < 3:
        raise RuntimeError(f"Not enough valid points left ({len(filtered_df)})")

    x = filtered_df[x_col].to_numpy()
    y = filtered_df[y_col].to_numpy()
    yerr = filtered_df[yerr_col].to_numpy()

    # Dynamic parameter estimation
    min_x, max_x = np.min(x), np.max(x)
    min_y, max_y = np.min(y), np.max(y)

    a0 = max(0.0, float(min_y))
    c0 = 1.5
    x_diff = (max_x**c0 - min_x**c0)
    b0 = max(1e-3, float((max_y - min_y) / x_diff)) if x_diff > 0 else 1.0

    # Expand upper bounds to accommodate larger y scales dynamically
    max_a_bound = max(5.0 * max_y, 10.0)
    bounds = (
        (0.0, 1e-6, 0.01),             # Lower bounds
        (max_a_bound, 1e6, 10.0)       # Dynamic upper bounds
    )

    # Force initial guess p0 to strictly lie inside bounds
    p0 = [
        np.clip(a0, bounds[0][0] + 1e-4, bounds[1][0] - 1e-4),
        np.clip(b0, bounds[0][1] + 1e-4, bounds[1][1] - 1e-4),
        np.clip(c0, bounds[0][2] + 1e-4, bounds[1][2] - 1e-4),
    ]

    try:
        popt, pcov = curve_fit(
            power_law,
            x,
            y,
            p0=p0,
            sigma=yerr,
            absolute_sigma=True,
            bounds=bounds,
            maxfev=50000,
        )
        perr = np.sqrt(np.diag(pcov))
        return popt, perr
    except Exception as e:
        raise RuntimeError(f"Fit failed: {e}")

In [ ]:
import numpy as np

from scipy.optimize import curve_fit

def exp_decay_plateau(x, a, b, c):
    """f(x) = a + b * exp(-x / c)
    Plateaus at LARGE x (used for Residual Range rr).
    """
    return a + b * np.exp(-x / c)


def eval_shift_fit(x, popt):
    """Evaluates the exponential-decay-plateau shift fit: a + b * exp(-x / c)."""
    return exp_decay_plateau(x, *popt)


def fit_shift_vs_mpv(df_res, x_col="rr_center", verbose=False):
    """Fits Shift ΔMPV vs a chosen x-variable (default: residual range) with
    a weighted exponential-decay-to-plateau model: shift(x) = a + b*exp(-x/c).
    """
    valid = df_res.dropna(subset=[x_col, "shift", "shift_err"])
    valid = valid[valid["shift_err"] > 0].sort_values(x_col)

    if len(valid) < 3:
        if verbose:
            print(f"  -> only {len(valid)} valid points, need >=3")
        return None, None

    x = valid[x_col].to_numpy()
    y = valid["shift"].to_numpy()
    yerr = valid["shift_err"].to_numpy()

    # Simple auto initial guess: plateau ~ last point, amplitude ~ first-last, decay ~ x-range/5
    a0 = y[-1]
    b0 = y[0] - a0
    c0 = max((x.max() - x.min()) / 5.0, 1e-3)
    p0 = [a0, b0, c0]

    try:
        popt, pcov = curve_fit(
            exp_decay_plateau,
            x,
            y,
            p0=p0,
            sigma=yerr,
            absolute_sigma=True,
            bounds=([-np.inf, -np.inf, 1e-6], [np.inf, np.inf, np.inf]),
            maxfev=20000,
        )
    except Exception as e:
        if verbose:
            print(f"  -> curve_fit failed: {e}, p0={p0}")
        return None, None

    perr = np.sqrt(np.diag(pcov))
    return popt, perr

def fit_rr_slices_theoretical(
    df,
    plane,
    tpc,
    hfit,
    pdg,
    dedx_col="dedx",
    rr_min=3.0,
    rr_max=40.0,
    rr_bin_width=1.0,
    hist_nbins=150,
    hist_xmin=0.0,
    hist_xmax=20.0,
    min_entries=30,
    pitch=0.32,
    mass=None,
    sigma0=0.2,
    sigma_bounds=(1e-4, 5.0),
    max_sigma_err=0.2,
    verbose=False,
):
    edges = make_rr_edges(rr_min, rr_max, rr_bin_width)
    rows = []

    for lo, hi in zip(edges[:-1], edges[1:]):
        sl = df[(df["rr"] >= lo) & (df["rr"] < hi)]
        if len(sl) < min_entries:
            continue
        rr_center = 0.5 * (lo + hi)

        pdf = build_theoretical_pdf(hfit, pdg, rr_center, pitch, mass=mass)
        mpv_theory = robust_max_x_py(pdf, 0.0, 10.0, 2000)
        x_grid, pdf_vals, dx = build_pdf_grid(pdf)

        counts, bin_edges = np.histogram(
            sl[dedx_col].to_numpy(),
            bins=hist_nbins,
            range=(hist_xmin, hist_xmax),
        )
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
        bin_errs = np.where(counts > 0, np.sqrt(counts), 1.0)

        # No seeding from a data-side peak finder here -- shift0=0.0 lets
        # curve_fit locate the shift purely from the fit itself, with no
        # correction toward a pre-measured data peak.
        fit = fit_theoretical_conv_slice(
            bin_centers,
            counts.astype(float),
            bin_errs,
            x_grid,
            pdf_vals,
            dx,
            mpv_theory,
            sigma0=sigma0,
            shift0=0.0,
            sigma_bounds=sigma_bounds,
        )

        if fit is None or fit["sigma_err"] > max_sigma_err:
            continue

        rows.append(
            dict(
                plane=plane,
                tpc=tpc,
                rr_center=rr_center,
                n_hits=len(sl),
                mpv_x=mpv_theory,
                mpv_theory=mpv_theory,
                gsigma=fit["sigma"],
                gsigma_err=fit["sigma_err"],
                shift=fit["shift"],
                shift_err=fit["shift_err"],
                fit_range_lo=fit["fit_range"][0],
                fit_range_hi=fit["fit_range"][1],
            )
        )

    return pd.DataFrame(rows)

def analyze_theoretical(
    hit_dfs,
    hfit,
    pdg,
    particle="muon",
    dedx_col="dedx",
    rr_max_by_particle=None,
    pitch=0.32,
    mass=None,
    sigma0=0.2,
    sigma_bounds=(1e-4, 5.0),
    max_sigma_err=0.2,
    out_prefix="langau_rr_theoretical",
    verbose=True,
):
    if rr_max_by_particle is None:
        rr_max_by_particle = {"muon": 80.0, "pion": 40.0, "proton": 60.0}
    rr_max = rr_max_by_particle.get(particle, 80.0)

    all_results = {}
    sigma_fit_params = {}
    shift_fit_params = {}

    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1, -1):
            if verbose:
                tpc_label = "combined" if tpc == -1 else tpc
                print(f"[theoretical] Plane {plane}, TPC {tpc_label}")

            sub = df if tpc == -1 else df[df["tpc"] == tpc]

            res = fit_rr_slices_theoretical(
                sub,
                plane,
                tpc,
                hfit=hfit,
                pdg=pdg,
                dedx_col=dedx_col,
                rr_max=rr_max,
                pitch=pitch,
                mass=mass,
                sigma0=sigma0,
                sigma_bounds=sigma_bounds,
                max_sigma_err=max_sigma_err,
                verbose=verbose,
            )
            all_results[(plane, tpc)] = res

            popt_sig = perr_sig = None
            popt_shf = perr_shf = None

            # In analyze_theoretical:
            if len(res) >= 3:
                try:
                    popt_sig, perr_sig = fit_sigma_vs_mpv(
                        res,
                        x_col="mpv_theory",
                        y_col="gsigma",
                        yerr_col="gsigma_err",
                    )
                except Exception:
                    popt_sig, perr_sig = None, None
                try:
                    # Use polynomial fit for shift parameters, vs residual range
                    popt_shf, perr_shf = fit_shift_vs_mpv(res, x_col="rr_center")
                except Exception:
                    popt_shf, perr_shf = None, None

            sigma_fit_params[(plane, tpc)] = (popt_sig, perr_sig)
            shift_fit_params[(plane, tpc)] = (popt_shf, perr_shf)

    non_empty = [
        r.assign(plane=p, tpc=t)
        for (p, t), r in all_results.items()
        if len(r)
    ]
    combined = (
        pd.concat(non_empty, ignore_index=True)
        if non_empty
        else pd.DataFrame()
    )

    if len(combined):
        combined.to_hdf(
            f"{out_prefix}_slices.h5",
            key="fits",
            mode="w",
            format="table",
            complib="blosc",
            complevel=9,
        )

    return all_results, sigma_fit_params, shift_fit_params

In [ ]:


all_results, sigma_fit_params,  shift_fit_params = analyze_theoretical(hit_dfs,hfit, pdg,  particle=particle)

In [ ]:
import numpy as np
from scipy.optimize import curve_fit

res = all_results[(0, -1)]  # pick one plane/tpc pair you know has data

valid = res.dropna(subset=["rr_center", "shift", "shift_err"])
valid = valid[valid["shift_err"] > 0]
valid = valid.sort_values("rr_center")

print("n valid:", len(valid))
print(valid[["rr_center", "shift", "shift_err"]])

x = valid["rr_center"].to_numpy()
y = valid["shift"].to_numpy()
yerr = valid["shift_err"].to_numpy()

n_edge = max(1, min(3, len(y) // 4))
a0 = np.median(y[-n_edge:])
b0 = np.median(y[:n_edge]) - a0
c0 = max((x.max() - x.min()) / 5.0, 1e-3)
p0 = [a0, b0, c0]
print("p0:", p0)

bounds = ([-np.inf, -np.inf, 1e-6], [np.inf, np.inf, np.inf])

# no try/except — let it raise
popt, pcov = curve_fit(
    exp_decay_plateau, x, y, p0=p0, sigma=yerr,
    absolute_sigma=True, bounds=bounds, maxfev=20000,
)
print("popt:", popt)

In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

PLANE_COLORS = {0: "tab:red", 1: "tab:blue", 2: "tab:green"}
TPC_STYLES = {
    0: {"linestyle": "--", "marker": "o", "label_prefix": "x < 0 (TPC 0)"},
    1: {"linestyle": ":", "marker": "s", "label_prefix": "x > 0 (TPC 1)"},
    -1: {"linestyle": "-", "marker": "^", "label_prefix": "Combined"},
}

'''
def plot_all_planes_shift(all_results, shift_fit_params, out_prefix):
    """Plots all (plane, tpc) polynomial shift curves on a single plot."""
    fig, ax = plt.subplots(figsize=(8, 8))

    for (plane, tpc), res in all_results.items():
        popt, _ = shift_fit_params.get((plane, tpc), (None, None))
        if popt is None or res is None or not len(res):
            continue

        color = PLANE_COLORS.get(plane, "black")
        style_info = TPC_STYLES.get(
            tpc, {"linestyle": "-", "label_prefix": f"tpc={tpc}"}
        )

        xs = np.linspace(res["mpv_theory"].min(), res["mpv_theory"].max(), 200)
        ys = eval_shift_fit(xs, popt)

        # Format label dynamically for quadratic polynomial: p[0]*x^2 + p[1]*x + p[2]
        if len(popt) == 3:
            label = (
                f"Plane {plane} ({style_info['label_prefix']}): "
                f"{popt[0]:+.2e}x² {popt[1]:+.2e}x {popt[2]:+.2e}"
            )
        else:
            label = f"Plane {plane} ({style_info['label_prefix']}) Fit"

        ax.plot(
            xs,
            ys,
            linestyle=style_info["linestyle"],
            color=color,
            linewidth=2.0,
            label=label,
        )

    ax.set_xlabel("Theoretical MPV [MeV/cm]", fontsize=12)
    ax.set_ylabel(r"Shift $\Delta$MPV [MeV/cm]", fontsize=12)
    ax.legend(fontsize=9, loc="lower left", framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes_shift.pdf")
    plt.show()
'''

def plot_all_planes_shift(all_results, shift_fit_params, out_prefix):
    """Plots all (plane, tpc) polynomial shift curves vs residual range on a single plot."""
    fig, ax = plt.subplots(figsize=(8, 8))

    for (plane, tpc), res in all_results.items():
        popt, _ = shift_fit_params.get((plane, tpc), (None, None))
        if popt is None or res is None or not len(res):
            continue

        color = PLANE_COLORS.get(plane, "black")
        style_info = TPC_STYLES.get(
            tpc, {"linestyle": "-", "label_prefix": f"tpc={tpc}"}
        )

        xs = np.linspace(res["rr_center"].min(), res["rr_center"].max(), 200)
        ys = eval_shift_fit(xs, popt)

        if len(popt) == 3:
            label = (
                f"Plane {plane} ({style_info['label_prefix']}): "
                f"{popt[0]:+.2e}x² {popt[1]:+.2e}x {popt[2]:+.2e}"
            )
        else:
            label = f"Plane {plane} ({style_info['label_prefix']}) Fit"

        ax.plot(
            xs,
            ys,
            linestyle=style_info["linestyle"],
            color=color,
            linewidth=2.0,
            label=label,
        )

    ax.set_xlabel("Residual Range [cm]", fontsize=12)
    ax.set_ylabel(r"Shift $\Delta$MPV [MeV/cm]", fontsize=12)
    ax.legend(fontsize=9, loc="lower left", framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.5)
    ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes_shift.pdf")
    plt.show()
def plot_all_planes(all_results, fit_params, particle, out_prefix):
    """Plots all (plane, tpc) power-law fit curves on a single square plot."""
    fig, ax = plt.subplots(figsize=(8, 8))

    for (plane, tpc), res in all_results.items():
        popt, _ = fit_params.get((plane, tpc), (None, None))
        if popt is None or res is None or not len(res):
            continue

        color = PLANE_COLORS[plane]
        style_info = TPC_STYLES.get(
            tpc,
            {"linestyle": "-", "label_prefix": f"tpc={tpc}"},
        )
        style = style_info["linestyle"]
        prefix = style_info["label_prefix"]

        xs = np.linspace(res["mpv_x"].min(), res["mpv_x"].max(), 200)
        label = (
            f"{prefix}, Plane {plane}: {popt[0]:.2f} + {popt[1]:.2f}x^{popt[2]:.2f}"
        )
        ax.plot(
            xs,
            power_law(xs, *popt),
            linestyle=style,
            color=color,
            linewidth=2.0,
            label=label,
        )

    ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
    ax.set_ylabel(r"$\sigma_G$ [MeV/cm]", fontsize=12)

    # Enlarged legend font size
    ax.legend(fontsize=10, loc="upper left", framealpha=0.9)
    ax.grid(True, linestyle=":", alpha=0.5)

    # Force square plot aspect ratio
    ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes.pdf")
    plt.show()


import matplotlib.pyplot as plt
import numpy as np

# Style mapping for consistency
TPC_STYLES = {
    0: {"linestyle": "--", "marker": "o", "label_prefix": "x < 0 (TPC 0)"},
    1: {"linestyle": ":", "marker": "s", "label_prefix": "x > 0 (TPC 1)"},
    -1: {"linestyle": "-", "marker": "^", "label_prefix": "Combined"},
}


def plot_by_plane(
    all_results,
    fit_params,
    particle="muon",
    out_prefix="comparison",
    x_limits=None,  # Defaults to full range (None)
):
    """One square subplot per plane with TPC 0, 1, and Combined overlaid."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    tpc_colors = {0: "tab:red", 1: "tab:blue", -1: "tab:green"}

    for plane in range(3):
        ax = axes[plane]
        y_visible_min, y_visible_max = np.inf, -np.inf

        for tpc in [0, 1, -1]:
            res = all_results.get((plane, tpc))
            popt, _ = fit_params.get((plane, tpc), (None, None))

            if res is None or not len(res):
                continue

            color = tpc_colors[tpc]
            style_info = TPC_STYLES[tpc]
            style = style_info["linestyle"]
            marker = style_info["marker"]
            prefix = style_info["label_prefix"]

            # Filter data points if x_limits is provided
            if x_limits is not None:
                mask = (res["mpv_x"] >= x_limits[0]) & (
                    res["mpv_x"] <= x_limits[1]
                )
                res_filtered = res[mask]
            else:
                res_filtered = res

            # Plot data points
            if len(res_filtered):
                ax.errorbar(
                    res_filtered["mpv_x"],
                    res_filtered["gsigma"],
                    yerr=res_filtered["gsigma_err"],
                    fmt=marker,
                    color=color,
                    ms=4,
                    capsize=2,
                    label=f"{prefix} Data",
                )
                if x_limits is not None:
                    y_visible_min = min(
                        y_visible_min,
                        (
                            res_filtered["gsigma"] - res_filtered["gsigma_err"]
                        ).min(),
                    )
                    y_visible_max = max(
                        y_visible_max,
                        (
                            res_filtered["gsigma"] + res_filtered["gsigma_err"]
                        ).max(),
                    )

            # Plot fit curve
            if popt is not None:
                x_min = x_limits[0] if x_limits else res["mpv_x"].min()
                x_max = x_limits[1] if x_limits else res["mpv_x"].max()
                xs = np.linspace(x_min, x_max, 200)
                ys = power_law(xs, *popt)

                label_fit = f"{prefix}: {popt[0]:.2f} + {popt[1]:.2f}x^{popt[2]:.2f}"
                ax.plot(
                    xs,
                    ys,
                    linestyle=style,
                    color=color,
                    linewidth=1.8,
                    label=label_fit,
                )

                if x_limits is not None:
                    y_visible_min = min(y_visible_min, ys.min())
                    y_visible_max = max(y_visible_max, ys.max())

        # Apply zoom limits and adjust Y limits only when x_limits is explicitly specified
        if x_limits is not None:
            ax.set_xlim(x_limits)
            if not np.isinf(y_visible_min) and not np.isinf(y_visible_max):
                y_pad = (y_visible_max - y_visible_min) * 0.1
                ax.set_ylim(y_visible_min - y_pad, y_visible_max + y_pad)

        ax.set_title(f"Plane {plane}", fontsize=14, pad=10)
        ax.set_xlabel("MPV [MeV/cm]", fontsize=12)
        ax.set_ylabel(r"$\sigma_G$ [MeV/cm]", fontsize=12)

        ax.legend(fontsize=9, loc="upper left", framealpha=0.9)
        ax.grid(True, linestyle=":", alpha=0.5)

        # Enforce 1:1 square aspect ratio
        ax.set_box_aspect(1)

    fig.tight_layout()
    zoom_suffix = (
        f"_zoom_{x_limits[0]}_{x_limits[1]}".replace(".", "p")
        if x_limits
        else ""
    )
    fig.savefig(f"{out_prefix}_by_plane{zoom_suffix}.pdf")
    plt.show()

'''
def plot_by_plane_shift(
    all_results,
    shift_fit_params=None,
    out_prefix="shift_comparison",
    x_limits=None,
):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    tpc_colors = {0: "tab:red", 1: "tab:blue", -1: "tab:green"}

    for plane in range(3):
        ax = axes[plane]
        y_min, y_max = np.inf, -np.inf

        for tpc in [0, 1, -1]:
            res = all_results.get((plane, tpc))
            popt = (
                shift_fit_params.get((plane, tpc), (None, None))[0]
                if shift_fit_params
                else None
            )

            if res is None or not len(res):
                continue

            color = tpc_colors[tpc]
            style_info = TPC_STYLES[tpc]

            res_fil = (
                res[
                    (res["mpv_theory"] >= x_limits[0])
                    & (res["mpv_theory"] <= x_limits[1])
                ]
                if x_limits
                else res
            )

            if len(res_fil):
                ax.errorbar(
                    res_fil["mpv_theory"],
                    res_fil["shift"],
                    yerr=res_fil["shift_err"],
                    fmt=style_info["marker"],
                    color=color,
                    ms=4,
                    capsize=2,
                    label=f"{style_info['label_prefix']} Shift",
                )
                y_min = min(
                    y_min, (res_fil["shift"] - res_fil["shift_err"]).min()
                )
                y_max = max(
                    y_max, (res_fil["shift"] + res_fil["shift_err"]).max()
                )

            if popt is not None:
                x_min = x_limits[0] if x_limits else res["mpv_theory"].min()
                x_max = x_limits[1] if x_limits else res["mpv_theory"].max()
                xs = np.linspace(x_min, x_max, 200)
                ys = eval_shift_fit(xs, popt)

                ax.plot(
                    xs,
                    ys,
                    linestyle=style_info["linestyle"],
                    color=color,
                    linewidth=1.8,
                    label=f"{style_info['label_prefix']} Fit",
                )
                y_min, y_max = min(y_min, ys.min()), max(y_max, ys.max())

        if x_limits is not None:
            ax.set_xlim(x_limits)
            if not np.isinf(y_min) and not np.isinf(y_max):
                pad = (y_max - y_min) * 0.1
                ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_title(f"Plane {plane}", fontsize=14, pad=10)
        ax.set_xlabel("Theoretical MPV [MeV/cm]", fontsize=12)
        ax.set_ylabel(r"Shift $\Delta$MPV [MeV/cm]", fontsize=12)
        ax.legend(fontsize=9, loc="lower left", framealpha=0.9)
        ax.grid(True, linestyle=":", alpha=0.5)
        ax.set_box_aspect(1)

    fig.tight_layout()
    fig.savefig(f"{out_prefix}_shift_by_plane.pdf")
    plt.show()
'''

def plot_by_plane_shift(
    all_results,
    shift_fit_params=None,
    out_prefix="shift_comparison",
    x_limits=None,
):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    tpc_colors = {0: "tab:red", 1: "tab:blue", -1: "tab:green"}

    for plane in range(3):
        ax = axes[plane]
        y_min, y_max = np.inf, -np.inf

        for tpc in [0, 1, -1]:
            res = all_results.get((plane, tpc))
            popt = (
                shift_fit_params.get((plane, tpc), (None, None))[0]
                if shift_fit_params
                else None
            )

            if res is None or not len(res):
                continue

            color = tpc_colors[tpc]
            style_info = TPC_STYLES[tpc]

            res_fil = (
                res[
                    (res["rr_center"] >= x_limits[0])
                    & (res["rr_center"] <= x_limits[1])
                ]
                if x_limits
                else res
            )

            if len(res_fil):
                ax.errorbar(
                    res_fil["rr_center"],
                    res_fil["shift"],
                    yerr=res_fil["shift_err"],
                    fmt=style_info["marker"],
                    color=color,
                    ms=4,
                    capsize=2,
                    label=f"{style_info['label_prefix']} Shift",
                )
                y_min = min(
                    y_min, (res_fil["shift"] - res_fil["shift_err"]).min()
                )
                y_max = max(
                    y_max, (res_fil["shift"] + res_fil["shift_err"]).max()
                )

            if popt is not None:
                x_min = x_limits[0] if x_limits else res["rr_center"].min()
                x_max = x_limits[1] if x_limits else res["rr_center"].max()
                xs = np.linspace(x_min, x_max, 200)
                ys = eval_shift_fit(xs, popt)

                ax.plot(
                    xs,
                    ys,
                    linestyle=style_info["linestyle"],
                    color=color,
                    linewidth=1.8,
                    label=f"{style_info['label_prefix']} Fit",
                )
                y_min, y_max = min(y_min, ys.min()), max(y_max, ys.max())

        if x_limits is not None:
            ax.set_xlim(x_limits)
            if not np.isinf(y_min) and not np.isinf(y_max):
                pad = (y_max - y_min) * 0.1
                ax.set_ylim(y_min - pad, y_max + pad)

        ax.set_title(f"Plane {plane}", fontsize=14, pad=10)
        ax.set_xlabel("Residual Range [cm]", fontsize=12)
        ax.set_ylabel(r"Shift $\Delta$MPV [MeV/cm]", fontsize=12)
        ax.legend(fontsize=9, loc="lower left", framealpha=0.9)
        ax.grid(True, linestyle=":", alpha=0.5)
        ax.set_box_aspect(1)

    fig.tight_layout()
    zoom_suffix = (
        f"_zoom_{x_limits[0]}_{x_limits[1]}".replace(".", "p")
        if x_limits
        else ""
    )
    fig.savefig(f"{out_prefix}_shift_by_plane{zoom_suffix}.pdf")
    plt.show()

In [ ]:

plot_all_planes(
    all_results,
    sigma_fit_params,
    particle=particle,
    out_prefix="comparison",
)

plot_all_planes_shift(
    all_results,
    shift_fit_params,
    out_prefix="comparison",
)

plot_by_plane(
    all_results,
    sigma_fit_params,
    particle=particle,
    out_prefix="comparison"
)


plot_by_plane_shift(
    all_results,
    shift_fit_params,
    out_prefix="comparison",
)

plot_by_plane(
    all_results,
    sigma_fit_params,
    particle=particle,
    out_prefix="comparison",
    x_limits= [1.9, 3]
)


plot_by_plane_shift(
    all_results,
    shift_fit_params,
    out_prefix="comparison",
    x_limits= [1.9, 3]
)


In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np


def inspect_all_rr_slices(
    hit_dfs,
    rr_start=4,
    rr_end=40,
    rr_step=1,
    tpc=0,
    dedx_col="dedx",
    hist_nbins=150,
    hist_xmin=0.0,
    hist_xmax=10.0,
    first_stage_range=(0.5, 20.0),
    max_par_err=1.0,
):
    """Loops over 1 cm RR slices from rr_start to rr_end and plots Stage 1,

    Stage 2 window, and Stage 2 fit results across all planes.
    """
    n_planes = len(hit_dfs)
    rr_ranges = [(r, r + rr_step) for r in range(rr_start, rr_end, rr_step)]

    # Style definitions
    point_color = "#2b5c8f"
    window_color = "#8B0000"  # Dark red for the Stage 2 window

    for rr_min, rr_max in rr_ranges:
        rr_center = 0.5 * (rr_min + rr_max)

        fig = plt.figure(figsize=(8.5 * n_planes, 7.5))
        gs = gridspec.GridSpec(1, n_planes, wspace=0.25)

        x_eval = np.linspace(hist_xmin, hist_xmax, 400)
        max_y_global = 0.0
        axes = []

        for plane_idx, df in enumerate(hit_dfs):
            ax = fig.add_subplot(gs[0, plane_idx])
            axes.append(ax)

            if df is None:
                ax.set_title(
                    f"Plane {plane_idx}: No Data", fontsize=17, pad=10
                )
                continue

            sub = df if tpc == -1 else df[df["tpc"] == tpc]
            sub = sub[(sub["rr"] >= rr_min) & (sub["rr"] < rr_max)]
            if len(sub) < 10:
                ax.set_title(
                    f"Plane {plane_idx}: Low Stats ({len(sub)} hits)",
                    fontsize=17,
                    pad=10,
                )
                continue

            # Build ROOT histogram from selection
            hname = f"h_p{plane_idx}_rr{rr_center:.1f}"
            hist = th1_from_series(
                sub[dedx_col], hname, "", hist_nbins, hist_xmin, hist_xmax
            )

            bin_edges = np.linspace(hist_xmin, hist_xmax, hist_nbins + 1)
            bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
            counts = np.array(
                [hist.GetBinContent(i) for i in range(1, hist_nbins + 1)]
            )
            errors = np.array(
                [hist.GetBinError(i) for i in range(1, hist_nbins + 1)]
            )

            if len(counts) > 0:
                max_y_global = max(max_y_global, np.max(counts + errors))

            # 1. Plot Data (LaTeX formatted N_{hits})
            data_line = ax.errorbar(
                bin_centers,
                counts,
                yerr=errors,
                fmt="o",
                color=point_color,
                ecolor=point_color,
                elinewidth=1.0,
                alpha=0.6,
                markersize=3.5,
                capsize=1.5,
                label=rf"$N_{{\text{{hits}}}}$ = {len(sub)}",
            )

            # 2. Execute Fits
            fit_res = langau_fit_two_stage(
                hist,
                first_stage_range=first_stage_range,
                max_par_err=max_par_err,
            )

            s1_line = None
            s2_window_line = None
            s2_line = None

            if fit_res is not None:
                # --- Draw Stage 1 Fit ---
                f1 = fit_res.get("stage1_func")
                p1 = fit_res.get("stage1_pars")
                e1 = fit_res.get("stage1_errs")

                if f1 is not None and p1 is not None:
                    y_s1 = [f1.Eval(x) for x in x_eval]

                    mpv1_val, mpv1_err = p1[1], (e1[1] if e1 is not None else 0.0)
                    gsigma1_val = (
                        p1[3]
                        if len(p1) > 3
                        else (p1[2] if len(p1) > 2 else 0.0)
                    )
                    gsigma1_err = (
                        e1[3]
                        if (e1 is not None and len(e1) > 3)
                        else (e1[2] if (e1 is not None and len(e1) > 2) else 0.0)
                    )

                    label_s1 = (
                        f"Stage 1 Fit:\n"
                        f"  MPV = {mpv1_val:.2f} ± {mpv1_err:.2f}\n"
                        f"  $\sigma_G$ = {gsigma1_val:.2f} ± {gsigma1_err:.2f}"
                    )

                    (s1_line,) = ax.plot(
                        x_eval,
                        y_s1,
                        color="black",
                        linestyle="--",
                        linewidth=2.2,
                        alpha=0.85,
                        label=label_s1,
                    )

                # --- Draw Stage 2 Window Limits (Dark Red) ---
                s2_range = fit_res.get("stage2_range")
                if s2_range is not None:
                    s2_window_line = ax.axvline(
                        s2_range[0],
                        color=window_color,
                        linestyle=":",
                        linewidth=2.4,
                        label=f"S2 Window [{s2_range[0]:.2f}, {s2_range[1]:.2f}]",
                    )
                    ax.axvline(
                        s2_range[1],
                        color=window_color,
                        linestyle=":",
                        linewidth=2.4,
                    )

                # --- Draw Stage 2 Fit ---
                func = fit_res["func"]
                p2 = fit_res["pars"]
                e2 = fit_res["errs"]
                stage_used = fit_res.get("fit_stage", 2)

                mpv2_val, mpv2_err = p2[1], e2[1]
                gsigma2_val = (
                    p2[3] if len(p2) > 3 else (p2[2] if len(p2) > 2 else 0.0)
                )
                gsigma2_err = (
                    e2[3] if len(e2) > 3 else (e2[2] if len(e2) > 2 else 0.0)
                )

                fit_color = "tab:green" if stage_used == 2 else "purple"
                fit_style = "-" if stage_used == 2 else "-."
                stage_label = f"Stage {stage_used}"

                label_s2 = (
                    f"{stage_label} Fit:\n"
                    f"  MPV = {mpv2_val:.2f} ± {mpv2_err:.2f}\n"
                    f"  $\sigma_G$ = {gsigma2_val:.2f} ± {gsigma2_err:.2f}"
                )

                y_fit = [func.Eval(x) for x in x_eval]
                (s2_line,) = ax.plot(
                    x_eval,
                    y_fit,
                    color=fit_color,
                    linestyle=fit_style,
                    linewidth=2.6,
                    label=label_s2,
                )

            ax.set_title(f"Plane {plane_idx}", fontsize=17, pad=10)
            ax.set_xlabel(r"Hit $dE/dx$ [MeV/cm]", fontsize=16)
            ax.tick_params(axis="both", which="major", labelsize=12)
            ax.grid(True, linestyle=":", alpha=0.5)

            # Explicit re-ordering of legend handles and labels
            handles_raw = [s1_line, s2_line, s2_window_line, data_line]
            ordered_handles = [h for h in handles_raw if h is not None]
            ordered_labels = [h.get_label() for h in ordered_handles]

            ax.legend(
                ordered_handles,
                ordered_labels,
                loc="upper right",
                fontsize=14,
                framealpha=0.9,
                labelspacing=0.7,
                handlelength=2.0,
            )

            if plane_idx > 0:
                ax.set_ylabel("")
            else:
                ax.set_ylabel("Hits / Bin", fontsize=16)

            ax.set_box_aspect(1)

        # Synchronize Y-limits
        if max_y_global > 0:
            for ax in axes:
                ax.set_ylim(0, max_y_global * 1.35)

        tpc_str = f"TPC {tpc}" if tpc != -1 else "TPC Combined"
        fig.suptitle(
            rf"Langau Fits | {tpc_str} | Range: {rr_min} $\leq$ RR < {rr_max} cm",
            fontsize=19,
            y=1.02,
        )
        plt.tight_layout()
        plt.show()

In [ ]:
# Inputs
mc_hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
data_hit_dfs = [data_hit0_df, data_hit1_df, data_hit2_df]

# --- Run for MC (4 to 40 cm in 1 cm steps) ---
inspect_all_rr_slices(
    hit_dfs=data_hit_dfs,rr_start=4, rr_end=40, tpc=0
)
